# Lung Disease Prediction — Model Training
## VGG16 Transfer Learning on Chest X-Ray Pneumonia Dataset

**Instructions:** Just click **Runtime → Run all** (or Ctrl+F9). Everything runs automatically.

Make sure you have set **Runtime → Change runtime type → GPU (T4)** before running.

### Step 1: Install Dependencies

In [ ]:
!pip install -q tensorflow opencv-python-headless scikit-learn matplotlib kagglehub

### Step 2: Download Dataset from Kaggle

In [ ]:
import kagglehub
import os
import shutil

# Download dataset
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print(f"Dataset downloaded to: {path}")

# Find the chest_xray folder
for root, dirs, files in os.walk(path):
    if 'chest_xray' in dirs:
        DATA_DIR = os.path.join(root, 'chest_xray')
        break
    if 'train' in dirs and 'test' in dirs:
        DATA_DIR = root
        break

print(f"Data directory: {DATA_DIR}")
print(f"Contents: {os.listdir(DATA_DIR)}")

### Step 3: Import Libraries & Set Hyperparameters

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

# Paths
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
MODEL_SAVE_PATH = "vgg16_pneumonia.keras"

# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.0001
DROPOUT_RATE = 0.5
DENSE_UNITS = 512

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"Training images folder: {TRAIN_DIR}")
print(f"Test images folder: {TEST_DIR}")

### Step 4: Create Data Generators

In [ ]:
# Training data with augmentation + 15% validation split
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.15,
)

# Test data — only rescale, no augmentation
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True,
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples: {test_generator.samples}")
print(f"Classes: {train_generator.class_indices}")

### Step 5: Compute Class Weights (handles imbalanced data)

In [ ]:
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes,
)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

### Step 6: Build VGG16 Model

In [ ]:
# Load VGG16 pretrained on ImageNet (without the top classification layer)
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze all VGG16 layers — we only train our custom head
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(DENSE_UNITS, activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

### Step 7: Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True),
]

print("Starting training... This will take about 15-20 minutes with GPU.")
print("="*60)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict,
)

print("\nTraining complete!")

### Step 8: Evaluate on Test Set

In [ ]:
# Load best model
model = tf.keras.models.load_model(MODEL_SAVE_PATH)

# Evaluate
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# Classification report
y_pred_probs = model.predict(test_generator)
y_pred = (y_pred_probs >= 0.5).astype(int).flatten()
y_true = test_generator.classes

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]))

### Step 9: Plot Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title("Confusion Matrix", fontsize=16)
plt.colorbar(im)

classes = ["NORMAL", "PNEUMONIA"]
ax.set_xticks([0, 1])
ax.set_xticklabels(classes)
ax.set_yticks([0, 1])
ax.set_yticklabels(classes)

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=20)

ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix saved!")

### Step 10: Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"], label="Train Accuracy")
ax1.plot(history.history["val_accuracy"], label="Val Accuracy")
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()

ax2.plot(history.history["loss"], label="Train Loss")
ax2.plot(history.history["val_loss"], label="Val Loss")
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()
print("Training history plot saved!")

### Step 11: Download the Model & Images to Your Computer

Run this cell — it will trigger downloads of 3 files:
1. `vgg16_pneumonia.keras` — the trained model
2. `confusion_matrix.png`
3. `training_history.png`

In [ ]:
from google.colab import files

print("Downloading model file... (this may take a minute)")
files.download("vgg16_pneumonia.keras")

print("Downloading confusion matrix...")
files.download("confusion_matrix.png")

print("Downloading training history plot...")
files.download("training_history.png")

print("\nAll done! Check your browser's Downloads folder.")
print("Copy vgg16_pneumonia.keras into your project's model/ folder.")
print("Copy the .png files into your project's static/ folder.")